In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                            f1_score, roc_auc_score, classification_report)
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
import xgboost as xgb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

print("✓ Librerías importadas")

✓ Librerías importadas


In [7]:
print("="*70)
print("CARGANDO DATOS")
print("="*70)

train_df = pd.read_csv('train_format1.csv')
test_df = pd.read_csv('test_format1.csv')
user_info_df = pd.read_csv('user_info_format1.csv')
user_log_df = pd.read_csv('user_log_format1.csv', nrows=10000000)

print(f"\nDatos cargados:")
print(f"  Train: {len(train_df):,}")
print(f"  Test: {len(test_df):,}")
print(f"  User Log: {len(user_log_df):,}")

user_log_df.rename(columns={'seller_id': 'merchant_id'}, inplace=True)
relevant_users = set(train_df['user_id'].unique()) | set(test_df['user_id'].unique())
user_log_df = user_log_df[user_log_df['user_id'].isin(relevant_users)]

print(f"User log filtrado: {len(user_log_df):,}")

CARGANDO DATOS

Datos cargados:
  Train: 260,864
  Test: 261,477
  User Log: 10,000,000
User log filtrado: 9,999,008


In [8]:
print("\n" + "="*70)
print("ENCODING DE IDs")
print("="*70)

# Merge con user_info
train_df = train_df.merge(user_info_df, on='user_id', how='left')
test_df = test_df.merge(user_info_df, on='user_id', how='left')

# Rellenar NaNs en age_range y gender SI EXISTEN
if 'age_range' in train_df.columns:
    train_df['age_range'] = train_df['age_range'].fillna(0).astype(int)
    test_df['age_range'] = test_df['age_range'].fillna(0).astype(int)
else:
    train_df['age_range'] = 0
    test_df['age_range'] = 0

if 'gender' in train_df.columns:
    train_df['gender'] = train_df['gender'].fillna(2).astype(int)
    test_df['gender'] = test_df['gender'].fillna(2).astype(int)
else:
    train_df['gender'] = 2
    test_df['gender'] = 2

# Label encoding con manejo de valores desconocidos
le_user = LabelEncoder()
le_merchant = LabelEncoder()

train_df['user_encoded'] = le_user.fit_transform(train_df['user_id'])
train_df['merchant_encoded'] = le_merchant.fit_transform(train_df['merchant_id'])

# Transform para test (manejar valores desconocidos asignando 0)
def safe_transform(encoder, values):
    known_mask = values.isin(encoder.classes_)
    result = np.zeros(len(values), dtype=int)
    result[known_mask] = encoder.transform(values[known_mask])
    return result

test_df['user_encoded'] = safe_transform(le_user, test_df['user_id'])
test_df['merchant_encoded'] = safe_transform(le_merchant, test_df['merchant_id'])

n_users = train_df['user_encoded'].max() + 1
n_merchants = train_df['merchant_encoded'].max() + 1

print(f"\n✓ Encoding completado")
print(f"  Usuarios únicos:   {n_users:,}")
print(f"  Merchants únicos:  {n_merchants:,}")


ENCODING DE IDs

✓ Encoding completado
  Usuarios únicos:   212,046
  Merchants únicos:  2,001


In [12]:
print("\n" + "="*70)
print("SPLIT TRAIN/VAL Y MATRICES DE FEATURES")
print("="*70)

# Features que usas en los modelos (coinciden con lo que espera EmbeddingDataset)
features_basicas = ["user_encoded", "merchant_encoded", "age_range", "gender"]

# Matrices de features y target
X_basico = train_df[features_basicas].values
X_test_basico = test_df[features_basicas].values
y = train_df["label"].values

# Split train / validación
X_train_basico, X_val_basico, y_train, y_val = train_test_split(
    X_basico,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nShapes:")
print(f"  X_basico (train completo): {X_basico.shape}")
print(f"  X_test_basico (test):      {X_test_basico.shape}")
print(f"  X_train_basico:            {X_train_basico.shape}")
print(f"  X_val_basico:              {X_val_basico.shape}")
print(f"  y_train:                   {y_train.shape}")
print(f"  y_val:                     {y_val.shape}")

print("\nDistribución original en y_train:")
print(pd.Series(y_train).value_counts())
print(f"Ratio 0/1: {(y_train == 0).sum() / (y_train == 1).sum():.2f}:1")

# Si quieres seguir usando las variables balanceadas que tienes más abajo:
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_basico, y_train)

print("\nDistribución balanceada con SMOTE:")
print(pd.Series(y_train_balanced).value_counts())
print(f"Total balanced: {len(y_train_balanced):,}")



SPLIT TRAIN/VAL Y MATRICES DE FEATURES

Shapes:
  X_basico (train completo): (260864, 4)
  X_test_basico (test):      (261477, 4)
  X_train_basico:            (208691, 4)
  X_val_basico:              (52173, 4)
  y_train:                   (208691,)
  y_val:                     (52173,)

Distribución original en y_train:
0    195786
1     12905
Name: count, dtype: int64
Ratio 0/1: 15.17:1

Distribución balanceada con SMOTE:
0    195786
1    195786
Name: count, dtype: int64
Total balanced: 391,572


In [13]:
print("\n" + "="*70)
print("PREPARACION SIN BALANCEO (usando class_weight)")
print("="*70)

print("\nDistribución original:")
print(pd.Series(y_train).value_counts())
print(f"Ratio: {(y_train==0).sum() / (y_train==1).sum():.2f}:1")

# NO HACER BALANCEO - Usar datos originales
X_train_final = X_train_basico
y_train_final = y_train

print("\nUsando datos originales sin SMOTE")
print(f"Total samples: {len(y_train_final):,}")

# Calcular pesos de clase para la loss function
class_weights = compute_class_weight(
    'balanced', 
    classes=np.unique(y_train_final), 
    y=y_train_final
)
print(f"\nPesos de clase calculados: {class_weights}")
print(f"  Clase 0 (no compra): {class_weights[0]:.4f}")
print(f"  Clase 1 (compra):    {class_weights[1]:.4f}")
print(f"  Ratio de pesos: {class_weights[1]/class_weights[0]:.2f}:1")


PREPARACION SIN BALANCEO (usando class_weight)

Distribución original:
0    195786
1     12905
Name: count, dtype: int64
Ratio: 15.17:1

Usando datos originales sin SMOTE
Total samples: 208,691

Pesos de clase calculados: [0.5329569  8.08566447]
  Clase 0 (no compra): 0.5330
  Clase 1 (compra):    8.0857
  Ratio de pesos: 15.17:1


In [14]:
class EmbeddingDataset(Dataset):
    def __init__(self, X, y):
        self.user_ids = torch.LongTensor(X[:, 0])
        self.merchant_ids = torch.LongTensor(X[:, 1])
        self.age_range = torch.FloatTensor(X[:, 2])
        self.gender = torch.FloatTensor(X[:, 3])
        self.labels = torch.LongTensor(y)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return (self.user_ids[idx], self.merchant_ids[idx], 
                self.age_range[idx], self.gender[idx], self.labels[idx])

# USAR X_train_final en lugar de X_train_balanced
train_dataset = EmbeddingDataset(X_train_final, y_train_final)
val_dataset = EmbeddingDataset(X_val_basico, y_val)
test_dataset = EmbeddingDataset(X_test_basico, np.zeros(len(X_test_basico)))

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=512, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

print("✓ Datasets PyTorch creados")
print(f"  Train: {len(train_dataset):,} samples")
print(f"  Val:   {len(val_dataset):,} samples")
print(f"  Test:  {len(test_dataset):,} samples")

✓ Datasets PyTorch creados
  Train: 208,691 samples
  Val:   52,173 samples
  Test:  261,477 samples


In [15]:
print("\n" + "="*70)
print("VERIFICACION DE DATOS Y DATASETS")
print("="*70)

# 1. Verificar columnas en DataFrames
print("\n1. COLUMNAS EN DATAFRAMES:")
print(f"\nTrain DF ({len(train_df.columns)} columnas):")
print(train_df.columns.tolist())

print(f"\nTest DF ({len(test_df.columns)} columnas):")
print(test_df.columns.tolist())

# 2. Verificar shapes de arrays
print("\n2. SHAPES DE ARRAYS:")
print(f"X_basico (train):     {X_basico.shape}")
print(f"X_test_basico (test): {X_test_basico.shape}")
print(f"X_train_basico:       {X_train_basico.shape}")
print(f"X_val_basico:         {X_val_basico.shape}")
print(f"X_train_balanced:     {X_train_balanced.shape}")

# 3. Verificar features usadas
print("\n3. FEATURES BASICAS USADAS:")
print(f"Features: {features_basicas}")
print(f"Cantidad: {len(features_basicas)}")

# 4. Mostrar ejemplo de datos
print("\n4. EJEMPLO DE DATOS (primeras 5 filas):")
print("\nTrain DF (features básicas):")
print(train_df[features_basicas].head())

print("\nTest DF (features básicas):")
print(test_df[features_basicas].head())

# 5. Verificar rangos de valores
print("\n5. RANGOS DE VALORES:")
print(f"\nuser_encoded:")
print(f"  Min: {train_df['user_encoded'].min()}")
print(f"  Max: {train_df['user_encoded'].max()}")
print(f"  Únicos: {train_df['user_encoded'].nunique()}")

print(f"\nmerchant_encoded:")
print(f"  Min: {train_df['merchant_encoded'].min()}")
print(f"  Max: {train_df['merchant_encoded'].max()}")
print(f"  Únicos: {train_df['merchant_encoded'].nunique()}")

print(f"\nage_range:")
print(f"  Valores únicos: {sorted(train_df['age_range'].unique())}")

print(f"\ngender:")
print(f"  Valores únicos: {sorted(train_df['gender'].unique())}")

# 6. Verificar que no haya NaNs
print("\n6. VERIFICACION DE NaNs:")
print(f"\nTrain DF (features básicas):")
print(train_df[features_basicas].isna().sum())

print(f"\nTest DF (features básicas):")
print(test_df[features_basicas].isna().sum())

# 7. Verificar datasets PyTorch
print("\n7. DATASETS PYTORCH:")
print(f"Train dataset size:  {len(train_dataset)}")
print(f"Val dataset size:    {len(val_dataset)}")
print(f"Test dataset size:   {len(test_dataset)}")

# Verificar un batch
print("\n8. VERIFICAR UN BATCH DEL TRAIN LOADER:")
for user_ids, merchant_ids, age_range, gender, labels in train_loader:
    print(f"  Batch size: {len(user_ids)}")
    print(f"  user_ids shape: {user_ids.shape}")
    print(f"  merchant_ids shape: {merchant_ids.shape}")
    print(f"  age_range shape: {age_range.shape}")
    print(f"  gender shape: {gender.shape}")
    print(f"  labels shape: {labels.shape}")
    print(f"\n  Ejemplo de valores (primeros 3):")
    print(f"    user_ids: {user_ids[:3]}")
    print(f"    merchant_ids: {merchant_ids[:3]}")
    print(f"    age_range: {age_range[:3]}")
    print(f"    gender: {gender[:3]}")
    print(f"    labels: {labels[:3]}")
    break  # Solo verificar el primer batch

print("\n" + "="*70)
print("VERIFICACION COMPLETADA")
print("="*70)
print("\n✓ Todo listo para entrenar los modelos")


VERIFICACION DE DATOS Y DATASETS

1. COLUMNAS EN DATAFRAMES:

Train DF (7 columnas):
['user_id', 'merchant_id', 'label', 'age_range', 'gender', 'user_encoded', 'merchant_encoded']

Test DF (7 columnas):
['user_id', 'merchant_id', 'prob', 'age_range', 'gender', 'user_encoded', 'merchant_encoded']

2. SHAPES DE ARRAYS:
X_basico (train):     (260864, 4)
X_test_basico (test): (261477, 4)
X_train_basico:       (208691, 4)
X_val_basico:         (52173, 4)
X_train_balanced:     (391572, 4)

3. FEATURES BASICAS USADAS:
Features: ['user_encoded', 'merchant_encoded', 'age_range', 'gender']
Cantidad: 4

4. EJEMPLO DE DATOS (primeras 5 filas):

Train DF (features básicas):
   user_encoded  merchant_encoded  age_range  gender
0         16930              1528          6       0
1         16930                42          6       0
2         16930              1714          6       0
3         16930               862          6       0
4        115324              1924          0       0

Test DF (f

In [16]:
print("\n" + "="*70)
print("MODELO 1: GRAPH NEURAL NETWORK (GNN)")
print("="*70)

class SimpleGNN(nn.Module):
    def __init__(self, n_users, n_merchants, embedding_dim=64):
        super(SimpleGNN, self).__init__()
        
        self.user_embedding = nn.Embedding(n_users, embedding_dim)
        self.merchant_embedding = nn.Embedding(n_merchants, embedding_dim)
        
        # Demographics
        self.demo_fc = nn.Linear(2, 16)
        
        # Graph convolution layers
        self.gc1 = nn.Linear(embedding_dim * 2 + 16, 128)
        self.gc2 = nn.Linear(128, 64)
        self.gc3 = nn.Linear(64, 32)
        
        self.bn1 = nn.BatchNorm1d(128)
        self.bn2 = nn.BatchNorm1d(64)
        self.dropout = nn.Dropout(0.3)
        
        self.output = nn.Linear(32, 2)
        
        nn.init.xavier_uniform_(self.user_embedding.weight)
        nn.init.xavier_uniform_(self.merchant_embedding.weight)
    
    def forward(self, user_ids, merchant_ids, age_range, gender):
        user_emb = self.user_embedding(user_ids)
        merchant_emb = self.merchant_embedding(merchant_ids)
        
        # Demographics
        demo = torch.stack([age_range, gender], dim=1)
        demo_emb = torch.relu(self.demo_fc(demo))
        
        # Concatenar
        x = torch.cat([user_emb, merchant_emb, demo_emb], dim=1)
        
        x = self.gc1(x)
        x = self.bn1(x)
        x = torch.relu(x)
        x = self.dropout(x)
        
        x = self.gc2(x)
        x = self.bn2(x)
        x = torch.relu(x)
        x = self.dropout(x)
        
        x = self.gc3(x)
        x = torch.relu(x)
        
        x = self.output(x)
        return x

print("✓ Modelo GNN definido")


MODELO 1: GRAPH NEURAL NETWORK (GNN)
✓ Modelo GNN definido


In [17]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {device}")

gnn_model = SimpleGNN(n_users, n_merchants, embedding_dim=64).to(device)

weight_gnn = torch.tensor([1.0, class_weights[1]/class_weights[0]], 
                          dtype=torch.float32).to(device)
criterion_gnn = nn.CrossEntropyLoss(weight=weight_gnn)
optimizer_gnn = optim.Adam(gnn_model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler_gnn = optim.lr_scheduler.ReduceLROnPlateau(optimizer_gnn, mode='min', 
                                                      patience=5, factor=0.5)

best_val_loss = float('inf')
patience = 15
patience_counter = 0

print("\nEntrenando GNN...")
for epoch in range(100):
    gnn_model.train()
    train_loss = 0
    
    for user_ids, merchant_ids, age_range, gender, labels in train_loader:
        user_ids = user_ids.to(device)
        merchant_ids = merchant_ids.to(device)
        age_range = age_range.to(device)
        gender = gender.to(device)
        labels = labels.to(device)
        
        optimizer_gnn.zero_grad()
        outputs = gnn_model(user_ids, merchant_ids, age_range, gender)
        loss = criterion_gnn(outputs, labels)
        loss.backward()
        optimizer_gnn.step()
        
        train_loss += loss.item()
    
    gnn_model.eval()
    val_loss = 0
    with torch.no_grad():
        for user_ids, merchant_ids, age_range, gender, labels in val_loader:
            user_ids = user_ids.to(device)
            merchant_ids = merchant_ids.to(device)
            age_range = age_range.to(device)
            gender = gender.to(device)
            labels = labels.to(device)
            
            outputs = gnn_model(user_ids, merchant_ids, age_range, gender)
            loss = criterion_gnn(outputs, labels)
            val_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    scheduler_gnn.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(gnn_model.state_dict(), 'best_gnn_model.pth')
    else:
        patience_counter += 1
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}: Train={avg_train_loss:.4f}, Val={avg_val_loss:.4f}")
    
    if patience_counter >= patience:
        print(f"Early stopping en epoch {epoch+1}")
        break

gnn_model.load_state_dict(torch.load('best_gnn_model.pth'))
print("✓ GNN entrenado")

Dispositivo: cpu

Entrenando GNN...
Epoch 10: Train=0.0229, Val=4.2959
Early stopping en epoch 16
✓ GNN entrenado


In [18]:
gnn_model.eval()
gnn_predictions = []
gnn_predictions_proba = []

with torch.no_grad():
    for user_ids, merchant_ids, age_range, gender, _ in val_loader:
        user_ids = user_ids.to(device)
        merchant_ids = merchant_ids.to(device)
        age_range = age_range.to(device)
        gender = gender.to(device)
        
        outputs = gnn_model(user_ids, merchant_ids, age_range, gender)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)
        
        gnn_predictions.extend(preds.cpu().numpy())
        gnn_predictions_proba.extend(probs[:, 1].cpu().numpy())

print("RESULTADOS GNN:")
print(f"ROC-AUC:   {roc_auc_score(y_val, gnn_predictions_proba):.4f}")
print(f"Precision: {precision_score(y_val, gnn_predictions):.4f}")
print(f"Recall:    {recall_score(y_val, gnn_predictions):.4f}")
print(f"F1-Score:  {f1_score(y_val, gnn_predictions):.4f}")

RESULTADOS GNN:
ROC-AUC:   0.6462
Precision: 0.0937
Recall:    0.5902
F1-Score:  0.1617


In [19]:
print("\n" + "="*70)
print("MODELO 2: NEURAL EMBEDDINGS")
print("="*70)

class EmbeddingModel(nn.Module):
    def __init__(self, n_users, n_merchants, embedding_dim=50):
        super(EmbeddingModel, self).__init__()
        
        self.user_embedding = nn.Embedding(n_users, embedding_dim)
        self.merchant_embedding = nn.Embedding(n_merchants, embedding_dim)
        
        self.demo_fc = nn.Linear(2, 16)
        
        self.fc1 = nn.Linear(embedding_dim * 2 + 16, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.dropout1 = nn.Dropout(0.3)
        
        self.fc2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.dropout2 = nn.Dropout(0.3)
        
        self.fc3 = nn.Linear(64, 32)
        self.bn3 = nn.BatchNorm1d(32)
        self.dropout3 = nn.Dropout(0.2)
        
        self.output = nn.Linear(32, 2)
        
        nn.init.xavier_uniform_(self.user_embedding.weight)
        nn.init.xavier_uniform_(self.merchant_embedding.weight)
    
    def forward(self, user_ids, merchant_ids, age_range, gender):
        user_emb = self.user_embedding(user_ids)
        merchant_emb = self.merchant_embedding(merchant_ids)
        
        demo = torch.stack([age_range, gender], dim=1)
        demo_emb = torch.relu(self.demo_fc(demo))
        
        x = torch.cat([user_emb, merchant_emb, demo_emb], dim=1)
        
        x = self.fc1(x)
        x = self.bn1(x)
        x = torch.relu(x)
        x = self.dropout1(x)
        
        x = self.fc2(x)
        x = self.bn2(x)
        x = torch.relu(x)
        x = self.dropout2(x)
        
        x = self.fc3(x)
        x = self.bn3(x)
        x = torch.relu(x)
        x = self.dropout3(x)
        
        x = self.output(x)
        return x

print("✓ Modelo Embeddings definido")


MODELO 2: NEURAL EMBEDDINGS
✓ Modelo Embeddings definido


In [20]:
print("\nEntrenando Neural Embeddings...")

emb_model = EmbeddingModel(n_users, n_merchants, embedding_dim=50).to(device)

weight_emb = torch.tensor([1.0, class_weights[1]/class_weights[0]], 
                          dtype=torch.float32).to(device)
criterion_emb = nn.CrossEntropyLoss(weight=weight_emb)
optimizer_emb = optim.Adam(emb_model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler_emb = optim.lr_scheduler.ReduceLROnPlateau(optimizer_emb, mode='min', 
                                                      patience=5, factor=0.5)

best_val_loss_emb = float('inf')
patience_counter_emb = 0

for epoch in range(100):
    emb_model.train()
    train_loss = 0
    
    for user_ids, merchant_ids, age_range, gender, labels in train_loader:
        user_ids = user_ids.to(device)
        merchant_ids = merchant_ids.to(device)
        age_range = age_range.to(device)
        gender = gender.to(device)
        labels = labels.to(device)
        
        optimizer_emb.zero_grad()
        outputs = emb_model(user_ids, merchant_ids, age_range, gender)
        loss = criterion_emb(outputs, labels)
        loss.backward()
        optimizer_emb.step()
        
        train_loss += loss.item()
    
    emb_model.eval()
    val_loss = 0
    with torch.no_grad():
        for user_ids, merchant_ids, age_range, gender, labels in val_loader:
            user_ids = user_ids.to(device)
            merchant_ids = merchant_ids.to(device)
            age_range = age_range.to(device)
            gender = gender.to(device)
            labels = labels.to(device)
            
            outputs = emb_model(user_ids, merchant_ids, age_range, gender)
            loss = criterion_emb(outputs, labels)
            val_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    scheduler_emb.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss_emb:
        best_val_loss_emb = avg_val_loss
        patience_counter_emb = 0
        torch.save(emb_model.state_dict(), 'best_emb_model.pth')
    else:
        patience_counter_emb += 1
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}: Train={avg_train_loss:.4f}, Val={avg_val_loss:.4f}")
    
    if patience_counter_emb >= patience:
        print(f"Early stopping en epoch {epoch+1}")
        break

emb_model.load_state_dict(torch.load('best_emb_model.pth'))
print("✓ Neural Embeddings entrenado")


Entrenando Neural Embeddings...
Epoch 10: Train=0.0328, Val=2.8551
Early stopping en epoch 16
✓ Neural Embeddings entrenado


In [21]:
emb_model.eval()
emb_predictions = []
emb_predictions_proba = []

with torch.no_grad():
    for user_ids, merchant_ids, age_range, gender, _ in val_loader:
        user_ids = user_ids.to(device)
        merchant_ids = merchant_ids.to(device)
        age_range = age_range.to(device)
        gender = gender.to(device)
        
        outputs = emb_model(user_ids, merchant_ids, age_range, gender)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)
        
        emb_predictions.extend(preds.cpu().numpy())
        emb_predictions_proba.extend(probs[:, 1].cpu().numpy())

print("RESULTADOS NEURAL EMBEDDINGS:")
print(f"ROC-AUC:   {roc_auc_score(y_val, emb_predictions_proba):.4f}")
print(f"Precision: {precision_score(y_val, emb_predictions):.4f}")
print(f"Recall:    {recall_score(y_val, emb_predictions):.4f}")
print(f"F1-Score:  {f1_score(y_val, emb_predictions):.4f}")

RESULTADOS NEURAL EMBEDDINGS:
ROC-AUC:   0.6399
Precision: 0.0885
Recall:    0.6110
F1-Score:  0.1546


In [22]:
print("\n" + "="*70)
print("MODELO 3: DEEPFM")
print("="*70)

class DeepFM(nn.Module):
    def __init__(self, n_users, n_merchants, embedding_dim=32):
        super(DeepFM, self).__init__()
        
        # Embeddings para FM
        self.user_embedding_fm = nn.Embedding(n_users, embedding_dim)
        self.merchant_embedding_fm = nn.Embedding(n_merchants, embedding_dim)
        
        # Embeddings para DNN
        self.user_embedding_dnn = nn.Embedding(n_users, embedding_dim)
        self.merchant_embedding_dnn = nn.Embedding(n_merchants, embedding_dim)
        
        # Linear terms
        self.user_bias = nn.Embedding(n_users, 1)
        self.merchant_bias = nn.Embedding(n_merchants, 1)
        
        # Demographics
        self.demo_fc = nn.Linear(2, 16)
        
        # Deep layers
        dnn_input_dim = embedding_dim * 2 + 16
        self.dnn = nn.Sequential(
            nn.Linear(dnn_input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU()
        )
        
        self.output = nn.Linear(32, 2)
        
        # Init
        nn.init.xavier_uniform_(self.user_embedding_fm.weight)
        nn.init.xavier_uniform_(self.merchant_embedding_fm.weight)
        nn.init.xavier_uniform_(self.user_embedding_dnn.weight)
        nn.init.xavier_uniform_(self.merchant_embedding_dnn.weight)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.merchant_bias.weight)
    
    def forward(self, user_ids, merchant_ids, age_range, gender):
        # FM part
        user_emb_fm = self.user_embedding_fm(user_ids)
        merchant_emb_fm = self.merchant_embedding_fm(merchant_ids)
        
        # Interaction
        fm_interaction = torch.sum(user_emb_fm * merchant_emb_fm, dim=1, keepdim=True)
        
        # Linear
        linear = self.user_bias(user_ids) + self.merchant_bias(merchant_ids)
        
        # DNN part
        user_emb_dnn = self.user_embedding_dnn(user_ids)
        merchant_emb_dnn = self.merchant_embedding_dnn(merchant_ids)
        
        demo = torch.stack([age_range, gender], dim=1)
        demo_emb = torch.relu(self.demo_fc(demo))
        
        dnn_input = torch.cat([user_emb_dnn, merchant_emb_dnn, demo_emb], dim=1)
        dnn_output = self.dnn(dnn_input)
        
        # Output
        x = self.output(dnn_output)
        return x

print("✓ Modelo DeepFM definido")


MODELO 3: DEEPFM
✓ Modelo DeepFM definido


In [23]:
print("\nEntrenando DeepFM...")

deepfm_model = DeepFM(n_users, n_merchants, embedding_dim=32).to(device)

weight_deepfm = torch.tensor([1.0, class_weights[1]/class_weights[0]], 
                             dtype=torch.float32).to(device)
criterion_deepfm = nn.CrossEntropyLoss(weight=weight_deepfm)
optimizer_deepfm = optim.Adam(deepfm_model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler_deepfm = optim.lr_scheduler.ReduceLROnPlateau(optimizer_deepfm, mode='min', 
                                                         patience=5, factor=0.5)

best_val_loss_deepfm = float('inf')
patience_counter_deepfm = 0

for epoch in range(100):
    deepfm_model.train()
    train_loss = 0
    
    for user_ids, merchant_ids, age_range, gender, labels in train_loader:
        user_ids = user_ids.to(device)
        merchant_ids = merchant_ids.to(device)
        age_range = age_range.to(device)
        gender = gender.to(device)
        labels = labels.to(device)
        
        optimizer_deepfm.zero_grad()
        outputs = deepfm_model(user_ids, merchant_ids, age_range, gender)
        loss = criterion_deepfm(outputs, labels)
        loss.backward()
        optimizer_deepfm.step()
        
        train_loss += loss.item()
    
    deepfm_model.eval()
    val_loss = 0
    with torch.no_grad():
        for user_ids, merchant_ids, age_range, gender, labels in val_loader:
            user_ids = user_ids.to(device)
            merchant_ids = merchant_ids.to(device)
            age_range = age_range.to(device)
            gender = gender.to(device)
            labels = labels.to(device)
            
            outputs = deepfm_model(user_ids, merchant_ids, age_range, gender)
            loss = criterion_deepfm(outputs, labels)
            val_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    scheduler_deepfm.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss_deepfm:
        best_val_loss_deepfm = avg_val_loss
        patience_counter_deepfm = 0
        torch.save(deepfm_model.state_dict(), 'best_deepfm_model.pth')
    else:
        patience_counter_deepfm += 1
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}: Train={avg_train_loss:.4f}, Val={avg_val_loss:.4f}")
    
    if patience_counter_deepfm >= patience:
        print(f"Early stopping en epoch {epoch+1}")
        break

deepfm_model.load_state_dict(torch.load('best_deepfm_model.pth'))
print("✓ DeepFM entrenado")


Entrenando DeepFM...
Epoch 10: Train=0.0288, Val=2.7277
Early stopping en epoch 16
✓ DeepFM entrenado


In [24]:
deepfm_model.eval()
deepfm_predictions = []
deepfm_predictions_proba = []

with torch.no_grad():
    for user_ids, merchant_ids, age_range, gender, _ in val_loader:
        user_ids = user_ids.to(device)
        merchant_ids = merchant_ids.to(device)
        age_range = age_range.to(device)
        gender = gender.to(device)
        
        outputs = deepfm_model(user_ids, merchant_ids, age_range, gender)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)
        
        deepfm_predictions.extend(preds.cpu().numpy())
        deepfm_predictions_proba.extend(probs[:, 1].cpu().numpy())

print("RESULTADOS DEEPFM:")
print(f"ROC-AUC:   {roc_auc_score(y_val, deepfm_predictions_proba):.4f}")
print(f"Precision: {precision_score(y_val, deepfm_predictions):.4f}")
print(f"Recall:    {recall_score(y_val, deepfm_predictions):.4f}")
print(f"F1-Score:  {f1_score(y_val, deepfm_predictions):.4f}")

RESULTADOS DEEPFM:
ROC-AUC:   0.6392
Precision: 0.0917
Recall:    0.5812
F1-Score:  0.1584


In [25]:
print("\n" + "="*70)
print("MODELO: RANDOM FOREST")
print("="*70)

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("Entrenando Random Forest...")
rf_model.fit(X_train_balanced, y_train_balanced)

rf_predictions = rf_model.predict(X_val_basico)
rf_predictions_proba = rf_model.predict_proba(X_val_basico)[:, 1]

print("\nRESULTADOS RANDOM FOREST:")
print(f"ROC-AUC:   {roc_auc_score(y_val, rf_predictions_proba):.4f}")
print(f"Precision: {precision_score(y_val, rf_predictions):.4f}")
print(f"Recall:    {recall_score(y_val, rf_predictions):.4f}")
print(f"F1-Score:  {f1_score(y_val, rf_predictions):.4f}")


MODELO: RANDOM FOREST
Entrenando Random Forest...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:    8.4s
[Parallel(n_jobs=-1)]: Done 168 tasks      | elapsed:   48.4s
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:   53.8s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:    0.2s
[Parallel(n_jobs=16)]: Done 200 out of 200 | elapsed:    0.2s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s



RESULTADOS RANDOM FOREST:
ROC-AUC:   0.5075
Precision: 0.0644
Recall:    0.4107
F1-Score:  0.1113


[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:    0.2s
[Parallel(n_jobs=16)]: Done 200 out of 200 | elapsed:    0.2s finished


In [26]:
print("\n" + "="*70)
print("MODELO: LOGISTIC REGRESSION")
print("="*70)

lr_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)

print("Entrenando Logistic Regression...")
lr_model.fit(X_train_balanced, y_train_balanced)

lr_predictions = lr_model.predict(X_val_basico)
lr_predictions_proba = lr_model.predict_proba(X_val_basico)[:, 1]

print("\nRESULTADOS LOGISTIC REGRESSION:")
print(f"ROC-AUC:   {roc_auc_score(y_val, lr_predictions_proba):.4f}")
print(f"Precision: {precision_score(y_val, lr_predictions):.4f}")
print(f"Recall:    {recall_score(y_val, lr_predictions):.4f}")
print(f"F1-Score:  {f1_score(y_val, lr_predictions):.4f}")


MODELO: LOGISTIC REGRESSION
Entrenando Logistic Regression...

RESULTADOS LOGISTIC REGRESSION:
ROC-AUC:   0.5002
Precision: 0.0643
Recall:    0.6913
F1-Score:  0.1176


In [27]:
print("\n" + "="*70)
print("MODELO: XGBOOST")
print("="*70)

scale_pos_weight = (y_train_final  == 0).sum() / (y_train_final  == 1).sum()

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    eval_metric='auc'
)

print("Entrenando XGBoost...")
xgb_model.fit(X_train_final, y_train_final)

xgb_predictions = xgb_model.predict(X_val_basico)
xgb_predictions_proba = xgb_model.predict_proba(X_val_basico)[:, 1]

print("\nRESULTADOS XGBOOST:")
print(f"ROC-AUC:   {roc_auc_score(y_val, xgb_predictions_proba):.4f}")
print(f"Precision: {precision_score(y_val, xgb_predictions):.4f}")
print(f"Recall:    {recall_score(y_val, xgb_predictions):.4f}")
print(f"F1-Score:  {f1_score(y_val, xgb_predictions):.4f}")


MODELO: XGBOOST
Entrenando XGBoost...

RESULTADOS XGBOOST:
ROC-AUC:   0.5870
Precision: 0.0848
Recall:    0.4467
F1-Score:  0.1425


In [28]:
print("\n" + "="*70)
print("COMPARACION DE MODELOS")
print("="*70)

resultados = pd.DataFrame({
    'Modelo': ['GNN', 'Neural Embeddings', 'Random Forest', 'DeepFM', 'Logistic Regression', 'XGBoost'],
    'ROC-AUC': [
        roc_auc_score(y_val, gnn_predictions_proba),
        roc_auc_score(y_val, emb_predictions_proba),
        roc_auc_score(y_val, rf_predictions_proba),
        roc_auc_score(y_val, deepfm_predictions_proba),
        roc_auc_score(y_val, lr_predictions_proba),
        roc_auc_score(y_val, xgb_predictions_proba)
    ],
    'Precision': [
        precision_score(y_val, gnn_predictions),
        precision_score(y_val, emb_predictions),
        precision_score(y_val, rf_predictions),
        precision_score(y_val, deepfm_predictions),
        precision_score(y_val, lr_predictions),
        precision_score(y_val, xgb_predictions)
    ],
    'Recall': [
        recall_score(y_val, gnn_predictions),
        recall_score(y_val, emb_predictions),
        recall_score(y_val, rf_predictions),
        recall_score(y_val, deepfm_predictions),
        recall_score(y_val, lr_predictions),
        recall_score(y_val, xgb_predictions)
    ],
    'F1-Score': [
        f1_score(y_val, gnn_predictions),
        f1_score(y_val, emb_predictions),
        f1_score(y_val, rf_predictions),
        f1_score(y_val, deepfm_predictions),
        f1_score(y_val, lr_predictions),
        f1_score(y_val, xgb_predictions)
    ]
})

resultados = resultados.sort_values('ROC-AUC', ascending=False)
print(resultados.to_string(index=False))


COMPARACION DE MODELOS
             Modelo  ROC-AUC  Precision   Recall  F1-Score
                GNN 0.646201   0.093705 0.590205  0.161733
  Neural Embeddings 0.639942   0.088517 0.610973  0.154631
             DeepFM 0.639218   0.091665 0.581215  0.158355
            XGBoost 0.587027   0.084790 0.446683  0.142525
      Random Forest 0.507523   0.064358 0.410725  0.111279
Logistic Regression 0.500187   0.064265 0.691259  0.117597


In [29]:
print("\n" + "="*70)
print("PREDICCIONES EN TEST SET")
print("="*70)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. GNN
print("\n1. GNN...")
gnn_model.eval()
gnn_test_proba = []
with torch.no_grad():
    for user_ids, merchant_ids, age_range, gender, _ in test_loader:
        outputs = gnn_model(user_ids.to(device), merchant_ids.to(device), 
                           age_range.to(device), gender.to(device))
        probs = torch.softmax(outputs, dim=1)
        gnn_test_proba.extend(probs[:, 1].cpu().numpy())
gnn_test_proba = np.array(gnn_test_proba)
print(f"✓ {len(gnn_test_proba)} predicciones")

# 2. Neural Embeddings
print("\n2. Neural Embeddings...")
emb_model.eval()
emb_test_proba = []
with torch.no_grad():
    for user_ids, merchant_ids, age_range, gender, _ in test_loader:
        outputs = emb_model(user_ids.to(device), merchant_ids.to(device), 
                           age_range.to(device), gender.to(device))
        probs = torch.softmax(outputs, dim=1)
        emb_test_proba.extend(probs[:, 1].cpu().numpy())
emb_test_proba = np.array(emb_test_proba)
print(f"✓ {len(emb_test_proba)} predicciones")

# 3. DeepFM
print("\n3. DeepFM...")
deepfm_model.eval()
deepfm_test_proba = []
with torch.no_grad():
    for user_ids, merchant_ids, age_range, gender, _ in test_loader:
        outputs = deepfm_model(user_ids.to(device), merchant_ids.to(device), 
                              age_range.to(device), gender.to(device))
        probs = torch.softmax(outputs, dim=1)
        deepfm_test_proba.extend(probs[:, 1].cpu().numpy())
deepfm_test_proba = np.array(deepfm_test_proba)
print(f"✓ {len(deepfm_test_proba)} predicciones")

# 4. Random Forest
print("\n4. Random Forest...")
rf_test_proba = rf_model.predict_proba(X_test_basico)[:, 1]
print(f"✓ {len(rf_test_proba)} predicciones")

# 5. Logistic Regression
print("\n5. Logistic Regression...")
lr_test_proba = lr_model.predict_proba(X_test_basico)[:, 1]
print(f"✓ {len(lr_test_proba)} predicciones")

# 6. XGBoost
print("\n6. XGBoost...")
xgb_test_proba = xgb_model.predict_proba(X_test_basico)[:, 1]
print(f"✓ {len(xgb_test_proba)} predicciones")


PREDICCIONES EN TEST SET

1. GNN...
✓ 261477 predicciones

2. Neural Embeddings...
✓ 261477 predicciones

3. DeepFM...
✓ 261477 predicciones

4. Random Forest...


[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.1s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:    1.0s
[Parallel(n_jobs=16)]: Done 200 out of 200 | elapsed:    1.2s finished


✓ 261477 predicciones

5. Logistic Regression...
✓ 261477 predicciones

6. XGBoost...
✓ 261477 predicciones


In [30]:
import joblib
import torch

# DataFrames
train_df.to_parquet("train_df.parquet")
test_df.to_parquet("test_df.parquet")

# Resultados métricas
resultados.to_csv("resultados.csv", index=False)

# Encoders
joblib.dump(le_user, "le_user.pkl")
joblib.dump(le_merchant, "le_merchant.pkl")

# Modelos tradicionales
joblib.dump(rf_model, "rf_model.pkl")
joblib.dump(lr_model, "lr_model.pkl")
joblib.dump(xgb_model, "xgb_model.pkl")

# Modelos PyTorch (pesos)
torch.save(gnn_model.state_dict(), "gnn_model.pth")
torch.save(emb_model.state_dict(), "emb_model.pth")
torch.save(deepfm_model.state_dict(), "deepfm_model.pth")

# Si de verdad necesitas y_val y las proba:
import pickle

with open("validacion.pkl", "wb") as f:
    pickle.dump({
        "y_val": y_val,
        "gnn_predictions_proba": gnn_predictions_proba,
        "emb_predictions_proba": emb_predictions_proba,
        "deepfm_predictions_proba": deepfm_predictions_proba,
        "rf_predictions_proba": rf_predictions_proba,
        "lr_predictions_proba": lr_predictions_proba,
        "xgb_predictions_proba": xgb_predictions_proba,
    }, f)
